# Camada Silver: Limpeza e Qualidade

In [0]:

%run ./00_Pipeline_Football_Setup

# Setup Inicial do Ambiente
## Arquitetura Agnóstica (Cloud vs Local)

Este projeto foi construído para ser **100% agnóstico de infraestrutura**. Isso significa que a mesma regra de negócio desenvolvida em PySpark pode ser executada sem nenhuma alteração em:

1. **Nuvem Corporativa (Ex: Databricks, AWS e Azure):** Usando o Unity Catalog, Glue.
2. **Ambiente Local (Docker / Jupyter):** Salvando os arquivos localmente usando o formato aberto Delta Lake (`.save()`).

A variável `ENVIRONMENT` controla essa chave de forma transparente para as camadas Bronze, Silver e Gold. Altere-a dependendo da sua stack.

Ambiente configurado: databricks


In [0]:
print("Lendo tabelas da Camada Bronze")

if ENVIRONMENT == "databricks":
    df_partidas_bruto = spark.table(f"{CATALOG_BRONZE}.matches")
    df_gols_bruto     = spark.table(f"{CATALOG_BRONZE}.goals_raw")
else:
    df_partidas_bruto = spark.read.format("delta").load(f"{PATH_BRONZE}/matches")
    df_gols_bruto     = spark.read.format("delta").load(f"{PATH_BRONZE}/goals_raw")

Lendo tabelas da Camada Bronze


In [0]:
print("Executando Quality Checks - Bronze")

total_partidas = df_partidas_bruto.count()

# QC-1: Campos obrigatorios ("Tem algum jogo sem data ou sem time?")
falhas_b1 = df_partidas_bruto.filter(
    F.col("match_date").isNull() | F.col("team_home").isNull()
    | F.col("team_away").isNull() | F.col("score_ft_home").isNull()
    | F.col("score_ft_away").isNull()
).count()
status_b1 = "Passou" if falhas_b1 == 0 else "Falhou"
print(f"QC-B1: Campos obrigatorios -> Status: {status_b1} (Falhas: {falhas_b1}/{total_partidas})")

# QC-2: Placar nao negativo ("Existe algum jogo com placar negativo, tipo -1 a 0?")
falhas_b2 = df_partidas_bruto.filter((F.col("score_ft_home") < 0) | (F.col("score_ft_away") < 0)).count()
status_b2 = "Passou" if falhas_b2 == 0 else "Falhou"
print(f"QC-B2: Placar nao negativo -> Status: {status_b2} (Falhas: {falhas_b2}/{total_partidas})")

Executando Quality Checks - Bronze
QC-B1: Campos obrigatorios -> Status: Passou (Falhas: 0/1066)
QC-B2: Placar nao negativo -> Status: Passou (Falhas: 0/1066)


In [0]:
# Tratamento de Partidas
df_partidas_quarentena = (
    df_partidas_bruto.filter(
        F.col("match_id").isNull()
        | F.col("match_date").isNull()
        | F.col("team_home").isNull()
        | F.col("team_away").isNull()
        | F.col("score_ft_home").isNull()
        | F.col("score_ft_away").isNull()
        | (F.col("score_ft_home") < 0)
        | (F.col("score_ft_away") < 0)
        | (F.upper(F.trim("team_home")) == F.upper(F.trim("team_away")))
    )
    .withColumn("quarantine_reason", F.lit("PARTIDA_INVALIDA"))
    .withColumn("quarantined_at", F.current_timestamp())
)

df_partidas_prata = (
    df_partidas_bruto.filter(
        F.col("match_id").isNotNull()
        & F.col("match_date").isNotNull()
        & F.col("team_home").isNotNull()
        & F.col("team_away").isNotNull()
        & F.col("score_ft_home").isNotNull()
        & F.col("score_ft_away").isNotNull()
        & (F.col("score_ft_home") >= 0)
        & (F.col("score_ft_away") >= 0)
        & (F.upper(F.trim("team_home")) != F.upper(F.trim("team_away")))
    )
    .select(
        "match_id",
        F.col("competition_name"),
        F.col("round").alias("matchday"),
        "match_date", "match_time",
        F.trim("team_home").alias("home_team"),
        F.trim("team_away").alias("away_team"),
        F.col("score_ht_home").cast("int").alias("home_score_ht"),
        F.col("score_ht_away").cast("int").alias("away_score_ht"),
        F.col("score_ft_home").cast("int").alias("home_score_ft"),
        F.col("score_ft_away").cast("int").alias("away_score_ft"),
        "source_file", "ingested_at"
    )
)

In [0]:
# Tratamento de Gols
df_gols_quarentena = (
    df_gols_bruto.filter(
        F.col("match_id").isNull()
        | F.col("team").isNull()
        | F.col("scorer").isNull()
        | F.col("minute").isNull()
        | (F.col("minute") < 0)
        | (F.col("minute") > 130)
    )
    .withColumn("quarantine_reason", F.lit("EVENTO_GOL_INVALIDO"))
    .withColumn("quarantined_at", F.current_timestamp())
)

df_gols_prata = (
    df_gols_bruto.filter(
        F.col("match_id").isNotNull()
        & F.col("team").isNotNull()
        & F.col("scorer").isNotNull()
        & F.col("minute").isNotNull()
        & (F.col("minute") >= 0)
        & (F.col("minute") <= 130)
    )
    .select(
        "match_id", "competition_name", "match_date",
        F.trim("team_home").alias("home_team"),
        F.trim("team_away").alias("away_team"),
        F.col("score_ft_home").cast("int").alias("home_score_ft"),
        F.col("score_ft_away").cast("int").alias("away_score_ft"),
        F.trim("team").alias("team"),
        F.trim("opponent").alias("opponent"),
        F.trim("scorer").alias("scorer"),
        F.col("minute").cast("int").alias("minute"),
        F.col("is_penalty").cast("int").alias("is_penalty"),
        F.col("is_own_goal").cast("int").alias("is_own_goal"),
        F.col("event_seq").cast("int").alias("event_seq"),
        "source_file", "ingested_at"
    )
)

# Como a API já atribui o gol para o time que GANHOU 
df_gols_prata = df_gols_prata.withColumn("credited_team", F.col("team"))
df_gols_prata = df_gols_prata.withColumn("conceding_team", F.col("opponent"))

In [0]:
tabelas_silver = {
    "matches": df_partidas_prata,
    "matches_quarantine": df_partidas_quarentena,
    "goal_events": df_gols_prata,
    "goal_events_quarantine": df_gols_quarentena,
}

for nome, df in tabelas_silver.items():
    if ENVIRONMENT == "databricks":
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG_SILVER}.{nome}")
    else:
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{PATH_SILVER}/{nome}")

print("Tabelas da Camada Silver salvas com sucesso.")

Tabelas da Camada Silver salvas com sucesso.


In [0]:
%sql
-- Tabela Fato:
-- Registra o o moto que a bola entra na rede, linha por gol
select * from workspace.silver.goal_events
limit 5


match_id,competition_name,match_date,home_team,away_team,home_score_ft,away_score_ft,team,opponent,scorer,minute,is_penalty,is_own_goal,event_seq,source_file,ingested_at,credited_team,conceding_team
0030aa35f001c27af1361f0158af3590,World Cup 2018,2018-07-07,Sweden,England,0,2,England,Sweden,Harry Maguire,30,0,0,1,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,2026-07-17T00:10:17.356Z,England,Sweden
0030aa35f001c27af1361f0158af3590,World Cup 2018,2018-07-07,Sweden,England,0,2,England,Sweden,Dele Alli,58,0,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,2026-07-17T00:10:17.356Z,England,Sweden
0040274278df02c89b2689b49bcc3bfa,World Cup 2018,2018-07-06,Uruguay,France,0,2,France,Uruguay,Raphaël Varane,40,0,0,1,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,2026-07-17T00:10:17.356Z,France,Uruguay
0040274278df02c89b2689b49bcc3bfa,World Cup 2018,2018-07-06,Uruguay,France,0,2,France,Uruguay,Antoine Griezmann,61,0,0,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2018/worldcup.json,2026-07-17T00:10:17.356Z,France,Uruguay
00bec281fafc7d47075d14d419616860,World Cup 1930,1930-07-17,Yugoslavia,Bolivia,4,0,Yugoslavia,Bolivia,Bek,60,0,0,1,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:17.356Z,Yugoslavia,Bolivia


In [0]:
%sql
-- Tabela Dimensao
-- Registra o resultado final do jogo, linha por partida
select * from workspace.silver.matches
limit 5

match_id,competition_name,matchday,match_date,match_time,home_team,away_team,home_score_ht,away_score_ht,home_score_ft,away_score_ft,source_file,ingested_at
1eda0860500cb7f1848fbd1e3d1c4e0d,World Cup 1930,Matchday 1,1930-07-13,null,France,Mexico,3,0,4,1,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:12.833Z
3b6e7a5527f7c35ff406c58adf9ed63b,World Cup 1930,Matchday 3,1930-07-15,null,Argentina,France,0,0,1,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:12.833Z
aaed4ce53e8c205b3beaa5291590abc9,World Cup 1930,Matchday 4,1930-07-16,null,Chile,Mexico,1,0,3,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:12.833Z
8ddf3688b8ba66ae57758461b19f267f,World Cup 1930,Matchday 7,1930-07-19,null,Chile,France,0,0,1,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:12.833Z
9c56f41e9c1389df6b49fe243f9c2923,World Cup 1930,Matchday 7,1930-07-19,null,Argentina,Mexico,3,1,6,3,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/1930/worldcup.json,2026-07-17T00:10:12.833Z


In [0]:
%sql
-- Dados da tabelas de Quarentena matches demonstra que todos os jogos passaram na Data Quality
-- Exemplo: data preenchida, nomes dos times válidos e placares não negativos
select * from workspace.silver.matches_quarantine
limit 5

competition_name,source_file,round,match_date,match_time,team_home,team_away,group,ground,score_ft_home,score_ft_away,score_ht_home,score_ht_away,match_id,ingested_at,quarantine_reason,quarantined_at


In [0]:
%sql
-- Dados da tabela de Quarentena com o motivo 'EVENTO_GOL_INVALIDO' informa que os gols nao teve os minutos registrados
-- Exemplo: Gols do Neymar e Haaland
select * from workspace.silver.goal_events_quarantine
limit 5

match_id,competition_name,match_date,team_home,team_away,score_ft_home,score_ft_away,source_file,team,opponent,scorer,minute,is_penalty,is_own_goal,ingested_at,event_seq,quarantine_reason,quarantined_at
03b3411d2c863c22f08a43de17254b3d,World Cup 2026,2026-06-14,Ivory Coast,Ecuador,1,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2026/worldcup.json,Ivory Coast,Ecuador,Amad Diallo,null,0,0,2026-07-17T00:10:17.356Z,1,EVENTO_GOL_INVALIDO,2026-07-17T00:10:49.920Z
04732b9ad8df6474043b1e06142a9e08,World Cup 2026,2026-06-24,South Africa,South Korea,1,0,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2026/worldcup.json,South Africa,South Korea,Thapelo Maseko,null,0,0,2026-07-17T00:10:17.356Z,1,EVENTO_GOL_INVALIDO,2026-07-17T00:10:49.920Z
049d1e9862e9f263478513eb65f12231,World Cup 2026,2026-07-05,Brazil,Norway,1,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2026/worldcup.json,Brazil,Norway,Neymar,null,1,0,2026-07-17T00:10:17.356Z,1,EVENTO_GOL_INVALIDO,2026-07-17T00:10:49.920Z
049d1e9862e9f263478513eb65f12231,World Cup 2026,2026-07-05,Brazil,Norway,1,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2026/worldcup.json,Norway,Brazil,Erling Haaland,null,0,0,2026-07-17T00:10:17.356Z,2,EVENTO_GOL_INVALIDO,2026-07-17T00:10:49.920Z
049d1e9862e9f263478513eb65f12231,World Cup 2026,2026-07-05,Brazil,Norway,1,2,dbfs:/Volumes/workspace/bronze/football_raw/json_incremental/worldcup.json/worldcup.json-master/2026/worldcup.json,Norway,Brazil,Erling Haaland,null,0,0,2026-07-17T00:10:17.356Z,3,EVENTO_GOL_INVALIDO,2026-07-17T00:10:49.920Z


In [0]:
# Auditoria Data Quality:

df_gols_prata.createOrReplaceTempView("vw_silver_goals")
print("\n  FINAL DA COPA DE 2022 (Argentina x França)")
print("-" * 80)
spark.sql("""
    SELECT 
        minute as Minuto,
        scorer as Jogador,
        credited_team as Time_que_Marcou,
        CASE WHEN is_penalty = 1 THEN 'Sim' ELSE 'Não' END as Foi_Penalti
    FROM vw_silver_goals
    WHERE competition_name = 'World Cup 2022' 
      AND (credited_team = 'Argentina' OR credited_team = 'France')
      AND match_date = '2022-12-18'
    ORDER BY minute ASC, event_seq ASC
""").show(truncate=False)
print("Se você está vendo a sequência exata de gols do Messi e Mbappé, nossa Camada Silver está validada com sucesso!")



  FINAL DA COPA DE 2022 (Argentina x França)
--------------------------------------------------------------------------------
+------+--------------+---------------+-----------+
|Minuto|Jogador       |Time_que_Marcou|Foi_Penalti|
+------+--------------+---------------+-----------+
|23    |Lionel Messi  |Argentina      |Sim        |
|36    |Ángel Di María|Argentina      |Não        |
|80    |Kylian Mbappé |France         |Sim        |
|81    |Kylian Mbappé |France         |Não        |
|108   |Lionel Messi  |Argentina      |Não        |
|118   |Kylian Mbappé |France         |Sim        |
+------+--------------+---------------+-----------+

Se você está vendo a sequência exata de gols do Messi e Mbappé, nossa Camada Silver está validada com sucesso!
